In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel
import gradio as gr 

In [2]:
pdf_file=PdfReader(r"D:\Studying\ai_agents\1_foundations\quan\Quan Nguyen.pdf")
linkdin=""
for text in pdf_file.pages:
    if text:
        linkdin+=text.extract_text()

In [3]:
with open(r"D:\Studying\ai_agents\1_foundations\quan\Summary.txt","r") as f:
    summary=f.read()

In [4]:
name="Quan Nguyen"

In [5]:
system_prompt=f"""You are acting as {name} to answer anything relating to {name}, please be professional to answer as possible.
You will be given a summary of {name} and a linkdin profile of {name}, please answer as possible.
If you don't know the answer, please say "I don't know"
Here is the summary of {name}
{summary}
Here is the linkdin profile of {name}
{linkdin}
With the above information, please answer the question.
"""

In [6]:
openai=OpenAI()

In [7]:
def chat(message, history):
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=openai.chat.completions.create(model="gpt-4o-mini",messages=messages)
    return response.choices[0].message.content

In [10]:
gr.ChatInterface(chat,type='messages').launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [11]:
evalulation_system_prompt=f"""You are a professional evaluator to evaluate the answer of {name},
You are provided the conversation between user and an Agent.
You task is to evaluate the latest answer that is engaging professional and polite based on the summary and linkdin profile of {name}.
 Summary of {name}
 {summary}
 Here is the linkdin profile of {name}
 {linkdin}
 With the above information, please evaluate the answer.
 Please reply with wherether it is acceptable and provide your feedback
 """

In [ ]:
class Evaluation(BaseModel):
    is_acceptable:int
    feedback:str   

In [20]:
def evaluate_user_prompt(reply,message,history):
    evaluation_user_prompt=f"""
    Here is the conversation between user and an Agent: {history}
    Here is the lastest message from user: {message}
    Here is the latest reply from Agent: {reply}
    Please reply with wherether it is acceptable and provide your feedback
    """ 
    return evaluation_user_prompt

In [21]:
def eval(reply,message,history)->Evaluation:
    message=[{'role':'system','content':evalulation_system_prompt}]+[{'role':'user','content':evaluate_user_prompt(reply,message,history)}]
    response=openai.beta.chat.completions.parse(model="gpt-4o-mini",messages=message,response_format=Evaluation)
    return response.choices[0].message.parsed

In [28]:
def rerun(reply,feedback,message,history):
    updated_system_prompt=system_prompt + "\n The last respone is not acceptable due to quality check violation"
    updated_system_prompt+=f"\n The latest reply is {reply}"
    updated_system_prompt+=f"This is the reason for reject {feedback}"
    messages=[{'role':'system','content':updated_system_prompt}]+history+[{'role':'user','content':message}]
    response=openai.chat.completions.create(model="gpt-4o-mini",messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat_updated(message,history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages=[{'role':'system','content':system}]+history+[{'role':'user','content':message}]
    response=openai.chat.completions.create(model="gpt-4o-mini",messages=messages)
    reply=response.choices[0].message.content
    eval_variable=eval(response,message,history)
    if eval_variable.is_acceptable:
        print(f"Passed evaluation - returning reply")
    else:
         print(f"Failed evaluation - retrying")
         print(eval_variable.feedback)
         print(rerun(reply,eval_variable.feedback,message,history))
    return reply

In [ ]:
gr.ChatInterface(chat_updated,type='messages').launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply Hello! How can I assist you today?
Passed evaluation - returning reply Certainly! I am a passionate professional with a strong background in leveraging technology and data to help organizations achieve their goals. I graduated from the University of Economics Ho Chi Minh City in 2015 and began my career in advisory services at BDO and EY, where I developed my skills in creating innovative tools to streamline processes and improve efficiency.

Throughout my journey at companies like Sony and Lazada, I have honed my expertise in data visualization and analytics, recognizing the power of data to inform better decision-making. My skill set includes programming, building dashboards, and providing insights into business performance.

Currently, I am serving as a Section Manager at Sony Electronics Vietnam, where I lead enterprise planning, develop internal tools, and drive automation to enhance operational efficiency. I am dedicated to delivering high-qual